# Eval metaphor extraction approach on Danish data

Survey and other sources extracted metaphors are annotated for:
- metaphor true false
- metaphor useful for menu true false

In [58]:
survey_and_other = "resources/ignore/FÆLLES filtered_metaphor_data_SHARED_list.xlsx"
survey_and_other_tab = "FÆLLES LISTE"

survey_and_other_col_metaphor_b = "Metafor_or_not\nBettina"
survey_and_other_col_metaphor_menu_b = "Metafor_menu\nBettina"
survey_and_other_col_metaphor_m = "Metafor_or_not\nMette"
survey_and_other_col_metaphor_menu_m = "Metafor_menu\nMette"
survey_and_other_col_consensus = "Metafor_or_not\nSHARED"

In [ ]:
full_survey = "resources/ignore/dataset_2140253_20250308_15671279322069612044.xlsx"

In [59]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score, classification_report, confusion_matrix

df = pd.read_excel(f"../../{survey_and_other}", sheet_name=survey_and_other_tab)

# Drop rows where either annotator has no value for metaphor_or_not
col_b = survey_and_other_col_metaphor_b
col_m = survey_and_other_col_metaphor_m
col_consensus = survey_and_other_col_consensus

df_agree = df.dropna(subset=[col_b, col_m])

print(f"Total rows: {len(df)}")
print(f"Rows with both B and M annotations: {len(df_agree)}")
print(f"\nUnique values B: {df_agree[col_b].unique()}")
print(f"Unique values M: {df_agree[col_m].unique()}")
print(f"Unique values consensus: {df_agree[col_consensus].unique()}")

Total rows: 715
Rows with both B and M annotations: 709

Unique values B: ['No' 'Yes' 'yes' 'Yes ' 'no']
Unique values M: ['No' 'Yes' 'no' 'yes' 'yes, hvis pårørende' 'yes, pårørende' 'n o']
Unique values consensus: ['No' 'Yes' nan 'yes']


In [3]:
# Normalize annotations to binary yes/no
def normalize_metaphor(val):
    if pd.isna(val):
        return None
    val = str(val).strip().lower()
    if val.startswith("yes"):
        return "yes"
    elif val in ("no", "n o"):
        return "no"
    return None

df_agree = df.dropna(subset=[col_b, col_m]).copy()
df_agree["b_norm"] = df_agree[col_b].apply(normalize_metaphor)
df_agree["m_norm"] = df_agree[col_m].apply(normalize_metaphor)
df_agree["consensus_norm"] = df_agree[col_consensus].apply(normalize_metaphor)

# Drop rows that couldn't be normalized
df_agree = df_agree.dropna(subset=["b_norm", "m_norm"])

print(f"Rows after normalization: {len(df_agree)}")
print(f"\nValue counts B:\n{df_agree['b_norm'].value_counts()}")
print(f"\nValue counts M:\n{df_agree['m_norm'].value_counts()}")

Rows after normalization: 709

Value counts B:
b_norm
yes    378
no     331
Name: count, dtype: int64

Value counts M:
m_norm
yes    377
no     332
Name: count, dtype: int64


In [4]:
# Agreement scores between B and M on metaphor or not
from sklearn.metrics import cohen_kappa_score, confusion_matrix

b = df_agree["b_norm"]
m = df_agree["m_norm"]

# Percent agreement
pct_agreement = (b == m).mean()

# Cohen's kappa
kappa = cohen_kappa_score(b, m)

# Confusion matrix
labels = ["yes", "no"]
cm = confusion_matrix(b, m, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"B={l}" for l in labels], columns=[f"M={l}" for l in labels])

print("=== Inter-Annotator Agreement (B vs M): Metaphor or Not ===\n")
print(f"Percent agreement: {pct_agreement:.4f} ({pct_agreement*100:.1f}%)")
print(f"Cohen's kappa:     {kappa:.4f}")
print(f"\nConfusion matrix:")
print(cm_df)

# Also compute agreement of each annotator with consensus
df_cons = df_agree.dropna(subset=["consensus_norm"])
print(f"\n\n=== Agreement with Consensus (n={len(df_cons)}) ===\n")

kappa_b_cons = cohen_kappa_score(df_cons["b_norm"], df_cons["consensus_norm"])
pct_b_cons = (df_cons["b_norm"] == df_cons["consensus_norm"]).mean()
print(f"B vs Consensus:  kappa={kappa_b_cons:.4f}, agreement={pct_b_cons*100:.1f}%")

kappa_m_cons = cohen_kappa_score(df_cons["m_norm"], df_cons["consensus_norm"])
pct_m_cons = (df_cons["m_norm"] == df_cons["consensus_norm"]).mean()
print(f"M vs Consensus:  kappa={kappa_m_cons:.4f}, agreement={pct_m_cons*100:.1f}%")

=== Inter-Annotator Agreement (B vs M): Metaphor or Not ===

Percent agreement: 0.9986 (99.9%)
Cohen's kappa:     0.9972

Confusion matrix:
       M=yes  M=no
B=yes    377     1
B=no       0   331


=== Agreement with Consensus (n=708) ===

B vs Consensus:  kappa=0.9972, agreement=99.9%
M vs Consensus:  kappa=0.9943, agreement=99.7%


In [5]:
# All rows are extracted by the metaphor extractor (all predicted as metaphor).
# "No" annotations = extractor was wrong (false positives).

print("=== Extractor Precision (all extracted = predicted metaphor) ===\n")

n = len(df_agree)
b_no = (df_agree["b_norm"] == "no").sum()
m_no = (df_agree["m_norm"] == "no").sum()

print(f"According to B:  {b_no}/{n} not a metaphor ({b_no/n*100:.1f}%) → precision = {(n-b_no)/n*100:.1f}%")
print(f"According to M:  {m_no}/{n} not a metaphor ({m_no/n*100:.1f}%) → precision = {(n-m_no)/n*100:.1f}%")

df_cons2 = df_agree.dropna(subset=["consensus_norm"])
n_cons = len(df_cons2)
cons_no = (df_cons2["consensus_norm"] == "no").sum()
print(f"Consensus:       {cons_no}/{n_cons} not a metaphor ({cons_no/n_cons*100:.1f}%) → precision = {(n_cons-cons_no)/n_cons*100:.1f}%")

=== Extractor Precision (all extracted = predicted metaphor) ===

According to B:  331/709 not a metaphor (46.7%) → precision = 53.3%
According to M:  332/709 not a metaphor (46.8%) → precision = 53.2%
Consensus:       331/708 not a metaphor (46.8%) → precision = 53.2%


In [6]:
# Metaphors useful for menu
col_menu_b = survey_and_other_col_metaphor_menu_b
col_menu_m = survey_and_other_col_metaphor_menu_m

print("=== Metaphors Useful for Menu ===\n")
print(f"Unique values menu B: {df[col_menu_b].dropna().unique()}")
print(f"Unique values menu M: {df[col_menu_m].dropna().unique()}")

df["menu_b_norm"] = df[col_menu_b].apply(normalize_metaphor)
df["menu_m_norm"] = df[col_menu_m].apply(normalize_metaphor)

menu_b_yes = (df["menu_b_norm"] == "yes").sum()
menu_m_yes = (df["menu_m_norm"] == "yes").sum()
menu_b_total = df["menu_b_norm"].notna().sum()
menu_m_total = df["menu_m_norm"].notna().sum()

print(f"\nAccording to B:  {menu_b_yes}/{menu_b_total} useful for menu ({menu_b_yes/menu_b_total*100:.1f}%)")
print(f"According to M:  {menu_m_yes}/{menu_m_total} useful for menu ({menu_m_yes/menu_m_total*100:.1f}%)")

# Agreement on menu usefulness (where both annotated)
df_menu = df.dropna(subset=["menu_b_norm", "menu_m_norm"])
both_yes = ((df_menu["menu_b_norm"] == "yes") & (df_menu["menu_m_norm"] == "yes")).sum()
pct_menu_agree = (df_menu["menu_b_norm"] == df_menu["menu_m_norm"]).mean()
kappa_menu = cohen_kappa_score(df_menu["menu_b_norm"], df_menu["menu_m_norm"])

print(f"\nBoth B and M say useful: {both_yes}/{len(df_menu)}")
print(f"Menu agreement: {pct_menu_agree*100:.1f}%, kappa={kappa_menu:.4f}")

=== Metaphors Useful for Menu ===

Unique values menu B: ['No' 'Yes' 'yes']
Unique values menu M: ['No' 'Yes' 'no' 'yes']

According to B:  207/710 useful for menu (29.2%)
According to M:  208/711 useful for menu (29.3%)

Both B and M say useful: 206/709
Menu agreement: 99.7%, kappa=0.9932


In [60]:
# Unique source paths and categorization
print("=== Unique source_path values ===\n")
for i, sp in enumerate(df['source_path'].dropna().unique(), 1):
    print(f"  {i}. {sp}")

print(f"\n=== Unique source_file values ===\n")
for i, sf in enumerate(df['source_file'].dropna().unique(), 1):
    n = (df['source_file'] == sf).sum()
    print(f"  {i}. {sf}  (n={n})")

# Categorize by source type
def categorize_source(path):
    path = str(path).lower()
    if 'survey' in path or 'dataset_' in path:
        return 'survey'
    elif 'interview' in path:
        return 'interview'
    else:
        return 'other'

df['source_category'] = df['source_path'].apply(categorize_source)

print(f"\n=== Source categories ===\n")
for cat, grp in df.groupby('source_category'):
    files = grp['source_file'].unique()
    print(f"  {cat} (n={len(grp)}):")
    for f in files:
        n = (grp['source_file'] == f).sum()
        print(f"    - {f}  (n={n})")


=== Unique source_path values ===

  1. /work/speech/Metaphor data/Survey/dataset_2140253_20250308_15671279322069612044_long.xlsx
  2. /work/speech/Metaphor data/Transcriptions/Colorectal Cancer/interviews_colorectal_transcriptions_merged_merged_turns_20250731_162344.xlsx
  3. /work/speech/Metaphor data/Transcriptions/Mamma Cancer/interviews_mamma_transcriptions_merged_merged_turns_20250731_162244.xlsx

=== Unique source_file values ===

  1. dataset_2140253_20250308_15671279322069612044_long.xlsx  (n=377)
  2. interviews_colorectal_transcriptions_merged_merged_turns_20250731_162344.xlsx  (n=174)
  3. interviews_mamma_transcriptions_merged_merged_turns_20250731_162244.xlsx  (n=164)

=== Source categories ===

  interview (n=338):
    - interviews_colorectal_transcriptions_merged_merged_turns_20250731_162344.xlsx  (n=174)
    - interviews_mamma_transcriptions_merged_merged_turns_20250731_162244.xlsx  (n=164)
  survey (n=377):
    - dataset_2140253_20250308_15671279322069612044_long.xlsx

In [61]:
# === Precision & Menu usefulness per source category ===

# Ensure normalize_metaphor and needed columns exist on df
def normalize_metaphor(val):
    if pd.isna(val):
        return None
    val = str(val).strip().lower()
    if val.startswith("yes"):
        return "yes"
    elif val in ("no", "n o"):
        return "no"
    return None

col_b = survey_and_other_col_metaphor_b
col_m = survey_and_other_col_metaphor_m
col_consensus = survey_and_other_col_consensus
col_menu_b = survey_and_other_col_metaphor_menu_b
col_menu_m = survey_and_other_col_metaphor_menu_m

# Normalize all columns on df
for col, dest in [(col_b, "b_norm"), (col_m, "m_norm"), (col_consensus, "consensus_norm"),
                  (col_menu_b, "menu_b_norm"), (col_menu_m, "menu_m_norm")]:
    df[dest] = df[col].apply(normalize_metaphor)

print("=" * 70)
print("Precision & Menu usefulness per source category")
print("=" * 70)

for cat in sorted(df['source_category'].unique()):
    grp = df[df['source_category'] == cat]
    grp_agree = grp.dropna(subset=["b_norm", "m_norm"])
    n = len(grp_agree)
    
    print(f"\n--- {cat.upper()} (n={len(grp)}, annotated={n}) ---")
    
    # Precision (all extracted = predicted metaphor, "no" = false positive)
    if n > 0:
        b_yes = (grp_agree["b_norm"] == "yes").sum()
        m_yes = (grp_agree["m_norm"] == "yes").sum()
        print(f"  Precision B:         {b_yes}/{n} = {b_yes/n*100:.1f}%")
        print(f"  Precision M:         {m_yes}/{n} = {m_yes/n*100:.1f}%")
        
        grp_cons = grp_agree.dropna(subset=["consensus_norm"])
        if len(grp_cons) > 0:
            cons_yes = (grp_cons["consensus_norm"] == "yes").sum()
            print(f"  Precision consensus: {cons_yes}/{len(grp_cons)} = {cons_yes/len(grp_cons)*100:.1f}%")
    
    # Menu usefulness
    menu_grp = grp.dropna(subset=["menu_b_norm"])
    if len(menu_grp) > 0:
        menu_b_yes = (menu_grp["menu_b_norm"] == "yes").sum()
        print(f"  Menu useful B:       {menu_b_yes}/{len(menu_grp)} = {menu_b_yes/len(menu_grp)*100:.1f}%")
    
    menu_grp_m = grp.dropna(subset=["menu_m_norm"])
    if len(menu_grp_m) > 0:
        menu_m_yes = (menu_grp_m["menu_m_norm"] == "yes").sum()
        print(f"  Menu useful M:       {menu_m_yes}/{len(menu_grp_m)} = {menu_m_yes/len(menu_grp_m)*100:.1f}%")
    
    # Both agree on menu
    menu_both = grp.dropna(subset=["menu_b_norm", "menu_m_norm"])
    if len(menu_both) > 0:
        both_menu_yes = ((menu_both["menu_b_norm"] == "yes") & (menu_both["menu_m_norm"] == "yes")).sum()
        print(f"  Both say menu:       {both_menu_yes}/{len(menu_both)} = {both_menu_yes/len(menu_both)*100:.1f}%")


Precision & Menu usefulness per source category

--- INTERVIEW (n=338, annotated=337) ---
  Precision B:         88/337 = 26.1%
  Precision M:         88/337 = 26.1%
  Precision consensus: 88/336 = 26.2%
  Menu useful B:       5/338 = 1.5%
  Menu useful M:       6/337 = 1.8%
  Both say menu:       5/337 = 1.5%

--- SURVEY (n=377, annotated=372) ---
  Precision B:         290/372 = 78.0%
  Precision M:         289/372 = 77.7%
  Precision consensus: 289/372 = 77.7%
  Menu useful B:       202/372 = 54.3%
  Menu useful M:       202/374 = 54.0%
  Both say menu:       201/372 = 54.0%


## Interview metaphor extraction evaluation

**Setup:**
- Ground truth: `met_present` and `met_present MBJ` = two manual raters on 5 interview transcripts /Users/sanderputs/git/pic4dclean/resources/ignore/Interviews_mamma_transcriptions_merged_MBJ_BMK_10_12_2025.xlsx
- Full interviews (needs be down filtered to the 5) resources/ignore/interviews_mamma_transcriptions_merged.xlsx
- LLM predictions: /Users/sanderputs/git/pic4dclean/resources/ignore/FÆLLES filtered_metaphor_data_SHARED_list.xlsx
Goal: i need to know how good the LLM prediction for these 5 interviews are F1 recall etc.. to know details about recall is mainly important, list the missed ones

In [ ]:
import pandas as pd

# --- Paths ---
manual_path = "../../resources/ignore/Interviews_mamma_transcriptions_merged_MBJ_BMK_10_12_2025.xlsx"
merged_turns_path = "../../resources/ignore/interviews_mamma_transcriptions_merged.xlsx"
shared_path = "../../resources/ignore/FÆLLES filtered_metaphor_data_SHARED_list.xlsx"

# --- Explore ground truth structure ---
xl = pd.ExcelFile(manual_path)
print("=== Ground truth file ===")
print("Sheets:", xl.sheet_names)
for sheet in xl.sheet_names[:5]:
    df_tmp = xl.parse(sheet, nrows=3)
    print(f"\n--- Sheet: {sheet} ---")
    print(f"Shape: {df_tmp.shape}")
    print(f"Columns: {list(df_tmp.columns)}")


=== Ground truth file ===
Sheets: ['transcriptions', 'sanity_check']

--- Sheet: transcriptions ---
Shape: (3, 17)
Columns: ['file', 'segment_id', 'timestamp', 'speaker', 'text', 'met_present', 'met_present MBJ', 'Forklaring', 'tenor', 'vehicle', 'usas_tenor\xa0', 'usas_vehicle\xa0', 'met_explain\xa0', 'met_menu\xa0', 'met_menu_explain\xa0', 'cross_segment ', 'multiple_met\xa0']

--- Sheet: sanity_check ---
Shape: (0, 0)
Columns: []


In [29]:
# Explore ground truth data
df_ann = xl.parse("transcriptions")
print(f"Ground truth shape: {df_ann.shape}")
print(f"\nUnique files: {df_ann['file'].unique()}")
print(f"\nmet_present unique: {df_ann['met_present'].unique()}")
print(f"met_present MBJ unique: {df_ann['met_present MBJ'].unique()}")
print(f"\nmet_present value counts:\n{df_ann['met_present'].value_counts(dropna=False)}")
print(f"\nmet_present MBJ value counts:\n{df_ann['met_present MBJ'].value_counts(dropna=False)}")
print(f"\nSample rows:")
df_ann[['file', 'segment_id', 'text', 'met_present', 'met_present MBJ']].head(10)


Ground truth shape: (28340, 17)

Unique files: ['ID 37-transcription_anonymiseret.docx'
 'ID 66-transcription_anonymiseret.docx'
 'ID 09-transcription_edit færdig.docx'
 'ID 03-transcription_edit færdig.docx'
 'ID 62-transcription_anonymieret.docx'
 'ID 52-transcription_anonymiseret.docx'
 'ID 21-transcription_edit_færdig.docx'
 'ID 35-transcription_anonymiseret.docx'
 'ID 40-transcription_anonymiseret.docx'
 'ID 12-transcription_edit færdig.docx'
 'ID 71-transcription_edit_anonymiseret.docx'
 'ID 13-transcription_edit1_færdig.docx'
 'ID 39-transcription_anonymiseret.docx'
 'ID 08-transcription_edit færdig.docx'
 'ID 73-transcription_anonymiseret.docx'
 'ID 27-transcription_edit_færdig.docx'
 'ID 44-transcription_anonymiseret.docx'
 'ID 38-transcription_anonymiseret.docx'
 'ID 04-transcription_edit færdig.docx'
 'ID 01-transcription_edit færdig.docx'
 'ID 76-transcription_anonymiseret.docx'
 'ID 42-transcription_anonymiseret.docx'
 'ID 45-transcription_anonymiseret.docx'
 'ID 41-transc

,file,segment_id,text,met_present,met_present MBJ
0,ID 37-transcription_anonymiseret.docx,interviews_mamma_ID 37-transcription_anonymise...,Så optager den.,no,no
1,ID 37-transcription_anonymiseret.docx,interviews_mamma_ID 37-transcription_anonymise...,"Og så skriver vi, at det er den 24. april 2023,",no,no
2,ID 37-transcription_anonymiseret.docx,interviews_mamma_ID 37-transcription_anonymise...,"og vi har [patient navn] inde, som har CPR num...",no,no
3,ID 37-transcription_anonymiseret.docx,interviews_mamma_ID 37-transcription_anonymise...,"Jeg plejer altid at lægge ud med at høre, hvad...",no,no
4,ID 37-transcription_anonymiseret.docx,interviews_mamma_ID 37-transcription_anonymise...,Så jeg starter samme sted i går.,no,no
5,ID 37-transcription_anonymiseret.docx,interviews_mamma_ID 37-transcription_anonymise...,"Ikke så meget, at det er blevet hormonbehandli...",no,no
6,ID 37-transcription_anonymiseret.docx,interviews_mamma_ID 37-transcription_anonymise...,"Ja. Og hvad har de fortalt dig om, hvad de har...",no,no
7,ID 37-transcription_anonymiseret.docx,interviews_mamma_ID 37-transcription_anonymise...,Det har de ikke uddybet yderligere.,no,no
8,ID 37-transcription_anonymiseret.docx,interviews_mamma_ID 37-transcription_anonymise...,"De fleste af jer plejer at sige, at man har få...",no,no
9,ID 37-transcription_anonymiseret.docx,interviews_mamma_ID 37-transcription_anonymise...,"fordi vi skal fjerne knuden i rask væv, og der...",no,no


In [30]:
# Which files were annotated? (met_present is not NaN)
annotated = df_ann[df_ann['met_present'].notna()]
annotated_files = annotated['file'].unique()
print(f"Annotated files ({len(annotated_files)}):")
for f in annotated_files:
    n_segs = len(annotated[annotated['file'] == f])
    n_yes = (annotated[annotated['file'] == f]['met_present'] == 'yes').sum()
    print(f"  {f}: {n_segs} segments, {n_yes} metaphors")

print(f"\nTotal annotated segments: {len(annotated)}")
print(f"Total metaphors (met_present=yes): {(annotated['met_present'] == 'yes').sum()}")
print(f"Total metaphors (met_present MBJ=yes): {(annotated['met_present MBJ'] == 'yes').sum()}")

# Check agreement between the two raters
both = annotated[['met_present', 'met_present MBJ']].dropna(how='any')
agree = (both['met_present'] == both['met_present MBJ']).sum()
print(f"\nBoth raters annotated: {len(both)} segments")
print(f"Agreement: {agree}/{len(both)} ({agree/len(both)*100:.1f}%)")


Annotated files (5):
  ID 37-transcription_anonymiseret.docx: 865 segments, 17 metaphors
  ID 66-transcription_anonymiseret.docx: 913 segments, 6 metaphors
  ID 09-transcription_edit færdig.docx: 808 segments, 9 metaphors
  ID 03-transcription_edit færdig.docx: 936 segments, 3 metaphors
  ID 62-transcription_anonymieret.docx: 478 segments, 3 metaphors

Total annotated segments: 4000
Total metaphors (met_present=yes): 38
Total metaphors (met_present MBJ=yes): 38

Both raters annotated: 3999 segments
Agreement: 3999/3999 (100.0%)


In [35]:
# Load LLM predictions (FÆLLES LISTE = consensus sheet)
df_shared = xl_shared.parse("FÆLLES LISTE")
print(f"LLM predictions shape: {df_shared.shape}")
print(f"Unique source files: {df_shared['source_file'].nunique()}")
# Print just the unique source file names
for f in sorted(df_shared['source_file'].dropna().unique()):
    print(f"  {f}")


LLM predictions shape: (715, 27)
Unique source files: 3
  dataset_2140253_20250308_15671279322069612044_long.xlsx
  interviews_colorectal_transcriptions_merged_merged_turns_20250731_162344.xlsx
  interviews_mamma_transcriptions_merged_merged_turns_20250731_162244.xlsx


In [36]:
# Filter LLM predictions to mamma interviews only
df_llm_mamma = df_shared[df_shared['source_file'].str.contains('mamma', case=False, na=False)].copy()
print(f"Mamma LLM predictions: {df_llm_mamma.shape}")
print(f"\nColumns: {list(df_llm_mamma.columns)}")
print(f"\nsource_path unique values:")
for sp in df_llm_mamma['source_path'].dropna().unique()[:5]:
    print(f"  {sp}")
print(f"\nSample rows (key cols):")
print(df_llm_mamma[['original_row_index', 'original_text', 'metaphor_span']].head(5).to_string(max_colwidth=80))


Mamma LLM predictions: (164, 27)

Columns: ['original_row_index', 'source_file', 'source_path', 'speaker', 'original_text', 'has_error', 'error_message', 'Metafor_or_not\nSHARED', 'Metafor_or_not\nBettina', 'Metafor_menu\nBettina', 'Metafor_or_not\nMette', 'Metafor_menu\nMette', 'metaphor_number', 'metaphor_span', 'context', 'tenor_concept', 'tenor_usas_code', 'tenor_usas_description', 'tenor_surface', 'tenor_explicit', 'vehicle_concept', 'vehicle_surface', 'vehicle_basic_usas_code', 'vehicle_basic_usas_description', 'vehicle_contextual_usas_code', 'vehicle_contextual_usas_description', 'meaning_discrepancy']

source_path unique values:
  /work/speech/Metaphor data/Transcriptions/Mamma Cancer/interviews_mamma_transcriptions_merged_merged_turns_20250731_162244.xlsx

Sample rows (key cols):
     original_row_index                                                                    original_text               metaphor_span
551                9500                                            

In [37]:
# Check the merged turns file to understand the row index mapping
xl2 = pd.ExcelFile(merged_turns_path)
print("Merged turns sheets:", xl2.sheet_names)
df_f = xl2.parse(xl2.sheet_names[0])
print(f"Shape: {df_f.shape}")
print(f"Columns: {list(df_f.columns)}")
print(f"\nUnique files:")
if 'file' in df_f.columns:
    for f in df_f['file'].unique():
        print(f"  {f}")
elif 'source_file' in df_f.columns:
    for f in df_f['source_file'].unique():
        print(f"  {f}")


Merged turns sheets: ['transcriptions', 'sanity_check']
Shape: (28340, 5)
Columns: ['file', 'segment_id', 'timestamp', 'speaker', 'text']

Unique files:
  ID 37-transcription_anonymiseret.docx
  ID 66-transcription_anonymiseret.docx
  ID 09-transcription_edit færdig.docx
  ID 03-transcription_edit færdig.docx
  ID 62-transcription_anonymieret.docx
  ID 52-transcription_anonymiseret.docx
  ID 21-transcription_edit_færdig.docx
  ID 35-transcription_anonymiseret.docx
  ID 40-transcription_anonymiseret.docx
  ID 12-transcription_edit færdig.docx
  ID 71-transcription_edit_anonymiseret.docx
  ID 13-transcription_edit1_færdig.docx
  ID 39-transcription_anonymiseret.docx
  ID 08-transcription_edit færdig.docx
  ID 73-transcription_anonymiseret.docx
  ID 27-transcription_edit_færdig.docx
  ID 44-transcription_anonymiseret.docx
  ID 38-transcription_anonymiseret.docx
  ID 04-transcription_edit færdig.docx
  ID 01-transcription_edit færdig.docx
  ID 76-transcription_anonymiseret.docx
  ID 42-tra

In [40]:
# The original_row_index refers to a "merged_turns" version of the file (not the segment-level file).
# Strategy: match LLM predictions to ground truth segments by checking if the metaphor_span 
# appears in the segment text.

# Get the 5-file ground truth
df_5 = df_ann[df_ann['file'].isin(annotated_files)].copy()
print(f"Ground truth segments (5 files): {len(df_5)}")
print(f"Ground truth metaphors (met_present=yes): {(df_5['met_present'] == 'yes').sum()}")

# For each LLM prediction, find which ground truth segment(s) contain the metaphor_span
# First, match LLM original_text to determine which file each prediction belongs to
llm_to_file = {}
for idx, row in df_llm_mamma.iterrows():
    llm_text = str(row['original_text']).strip()
    # Search for this text (or a substring) in ground truth segments
    for f in annotated_files:
        file_segs = df_5[df_5['file'] == f]
        # Check if any segment's text is contained in (or contains) the LLM original_text
        mask = file_segs['text'].astype(str).apply(lambda t: t.strip() in llm_text or llm_text in t.strip())
        if mask.any():
            llm_to_file[idx] = f
            break

print(f"\nLLM predictions matched to annotated files: {len(llm_to_file)} / {len(df_llm_mamma)}")
for f in annotated_files:
    n = sum(1 for v in llm_to_file.values() if v == f)
    print(f"  {f}: {n}")


Ground truth segments (5 files): 4000
Ground truth metaphors (met_present=yes): 38

LLM predictions matched to annotated files: 81 / 164
  ID 37-transcription_anonymiseret.docx: 37
  ID 66-transcription_anonymiseret.docx: 12
  ID 09-transcription_edit færdig.docx: 15
  ID 03-transcription_edit færdig.docx: 15
  ID 62-transcription_anonymieret.docx: 2


In [41]:
# Match LLM predictions to specific ground truth segments using metaphor_span
# A segment is "LLM-predicted positive" if any LLM metaphor_span appears in it

llm_in_5 = df_llm_mamma.loc[list(llm_to_file.keys())].copy()
llm_in_5['matched_file'] = llm_in_5.index.map(llm_to_file)

# For each segment, check if any LLM metaphor_span appears in the segment text
matched_llm_indices = []
df_5 = df_5.copy()
df_5['llm_predicted'] = 0

for idx, row in llm_in_5.iterrows():
    span = str(row['metaphor_span']).strip().lower()
    f = row['matched_file']
    file_mask = df_5['file'] == f
    text_mask = df_5.loc[file_mask, 'text'].astype(str).str.lower().str.contains(span, regex=False, na=False)
    matched_segs = df_5.loc[file_mask][text_mask].index
    if len(matched_segs) > 0:
        df_5.loc[matched_segs, 'llm_predicted'] = 1
        matched_llm_indices.append(idx)

llm_matched = llm_in_5.loc[llm_in_5.index.isin(matched_llm_indices)]
llm_unmatched = llm_in_5.loc[~llm_in_5.index.isin(matched_llm_indices)]

print(f"LLM predictions matched to specific segments: {len(llm_matched)} / {len(llm_in_5)}")
print(f"LLM predictions NOT matched to any segment: {len(llm_unmatched)}")
print(f"Segments flagged as LLM-predicted: {df_5['llm_predicted'].sum()}")


LLM predictions matched to specific segments: 18 / 81
LLM predictions NOT matched to any segment: 63
Segments flagged as LLM-predicted: 18


In [42]:
# Debug: Why are 63 LLM predictions unmatched?
# Check if the metaphor_span appears in merged original_text but not in individual segments
print("Sample unmatched LLM predictions:")
for i, (idx, row) in enumerate(llm_unmatched.head(5).iterrows()):
    span = str(row['metaphor_span']).strip()
    orig = str(row['original_text']).strip()
    f = row['matched_file']
    print(f"\n--- {i+1}. File: {f} ---")
    print(f"  metaphor_span: '{span}'")
    print(f"  original_text: '{orig[:120]}...'")
    print(f"  span in original_text: {span.lower() in orig.lower()}")
    
    # Find which segments make up this original_text
    file_segs = df_5[df_5['file'] == f]
    seg_matches = file_segs[file_segs['text'].astype(str).apply(lambda t: t.strip() in orig)]
    print(f"  Matching segments: {len(seg_matches)}")
    if len(seg_matches) > 0:
        combined = ' '.join(seg_matches['text'].astype(str).str.strip())
        print(f"  span in combined seg text: {span.lower() in combined.lower()}")


Sample unmatched LLM predictions:

--- 1. File: ID 37-transcription_anonymiseret.docx ---
  metaphor_span: 'heden svedetur'
  original_text: 'Så man kan så få hedet svedetur igen, og man kan opleve at kroppen sådan igen reagerer lidt mere, den føler sig gammel....'
  span in original_text: False
  Matching segments: 1
  span in combined seg text: False

--- 2. File: ID 66-transcription_anonymiseret.docx ---
  metaphor_span: 'der er sådan en blodprop i benet'
  original_text: 'Og når jeg snakker om den blodprop Så er der sådan en blodprop i benet...'
  span in original_text: False
  Matching segments: 2
  span in combined seg text: False

--- 3. File: ID 09-transcription_edit færdig.docx ---
  metaphor_span: 'boner ud til en behandling'
  original_text: 'Det første kort, jeg vil vise dig, det er sådan en tidslinje, hvor man lige kan få et overblik over, hvad er det egentli...'
  span in original_text: False
  Matching segments: 5
  span in combined seg text: False

--- 4. File: ID 37-tr

In [43]:
# Better approach: work at segment level with broader text matching
# For each ground truth segment, flag if ANY LLM prediction's original_text contains the segment text
# AND vice versa - for each LLM prediction, find which segments its original_text covers

# Step 1: For each ground truth "yes" segment, check if the LLM caught it
# A ground truth metaphor is "caught" if any LLM prediction's original_text contains
# the segment text (or overlaps sufficiently)

metaphor_segments = df_5[df_5['met_present'] == 'yes'].copy()
print(f"Ground truth metaphor segments: {len(metaphor_segments)}")

recovered = []
for seg_idx, seg_row in metaphor_segments.iterrows():
    seg_text = str(seg_row['text']).strip().lower()
    f = seg_row['file']
    found = False
    llm_match = ""
    
    for _, llm_row in llm_in_5[llm_in_5['matched_file'] == f].iterrows():
        orig = str(llm_row['original_text']).strip().lower()
        span = str(llm_row['metaphor_span']).strip().lower()
        
        # Check: segment text in LLM original_text, OR LLM original_text in segment text
        # OR metaphor_span in segment text
        if seg_text in orig or orig in seg_text or span in seg_text:
            found = True
            llm_match = str(llm_row['metaphor_span'])
            break
    
    recovered.append({
        'seg_idx': seg_idx,
        'file': f,
        'segment_id': seg_row['segment_id'],
        'text': str(seg_row['text']).strip()[:100],
        'met_present': seg_row['met_present'],
        'met_present_MBJ': seg_row['met_present MBJ'],
        'llm_found': found,
        'llm_span': llm_match
    })

df_recovery = pd.DataFrame(recovered)
tp = df_recovery['llm_found'].sum()
fn = (~df_recovery['llm_found']).sum()
total_gt = len(df_recovery)

print(f"\nRecall analysis (ground truth metaphor segments):")
print(f"  TP (LLM found): {tp}")
print(f"  FN (LLM missed): {fn}")
print(f"  Recall: {tp}/{total_gt} = {tp/total_gt*100:.1f}%")


Ground truth metaphor segments: 38

Recall analysis (ground truth metaphor segments):
  TP (LLM found): 4
  FN (LLM missed): 34
  Recall: 4/38 = 10.5%


In [51]:
# === SEGMENT-LEVEL EVALUATION ===
# Build y_true and y_pred at the segment level for the 5 annotated files.
#
# The LLM was run on merged turns (consecutive same-speaker segments merged).
# Match LLM predictions back to individual segments using:
#   1. File assignment via text containment
#   2. metaphor_span substring matching to segments
#   3. original_text containment (LLM merged turn contains segment text)
import re, unicodedata
from sklearn.metrics import classification_report, confusion_matrix, cohen_kappa_score
import numpy as np

def norm(t):
    t = unicodedata.normalize('NFC', str(t).strip())
    return re.sub(r'\s+', ' ', t).lower()

df_5 = df_ann[df_ann['file'].isin(annotated_files)].copy()

# Assign y_true from rater 1, rater 2, and union
df_5['y_true_r1'] = (df_5['met_present'] == 'yes').astype(int)
df_5['y_true_r2'] = (df_5['met_present MBJ'] == 'yes').astype(int)
df_5['y_true_union'] = ((df_5['met_present'] == 'yes') | (df_5['met_present MBJ'] == 'yes')).astype(int)

# Assign y_pred: flag segments that match any LLM prediction
df_5['y_pred'] = 0

for _, lr in llm_in_5.iterrows():
    f = lr['matched_file']
    span_n = norm(lr['metaphor_span'])
    orig_n = norm(lr['original_text'])
    file_mask = df_5['file'] == f
    
    for seg_idx in df_5[file_mask].index:
        seg_n = norm(df_5.loc[seg_idx, 'text'])
        # Match if: span in segment text, OR segment text in LLM original_text
        if span_n in seg_n or seg_n in orig_n:
            df_5.loc[seg_idx, 'y_pred'] = 1

# Also check unmapped LLM predictions by direct segment search
unmapped = df_llm_mamma.loc[~df_llm_mamma.index.isin(llm_in_5.index)]
for _, lr in unmapped.iterrows():
    span_n = norm(lr['metaphor_span'])
    orig_n = norm(lr['original_text'])
    for f in annotated_files:
        file_mask = df_5['file'] == f
        for seg_idx in df_5[file_mask].index:
            seg_n = norm(df_5.loc[seg_idx, 'text'])
            if span_n in seg_n or seg_n in orig_n:
                df_5.loc[seg_idx, 'y_pred'] = 1

n_pred = df_5['y_pred'].sum()
n_gt = df_5['y_true_r1'].sum()
print(f"Segments: {len(df_5)} | GT metaphors (R1): {n_gt} | LLM predicted: {n_pred}")


Segments: 4000 | GT metaphors (R1): 38 | LLM predicted: 118


In [52]:
# === CLASSIFICATION METRICS ===
y_true = df_5['y_true_r1']
y_pred = df_5['y_pred']

tp = ((y_true == 1) & (y_pred == 1)).sum()
fp = ((y_true == 0) & (y_pred == 1)).sum()
fn = ((y_true == 1) & (y_pred == 0)).sum()
tn = ((y_true == 0) & (y_pred == 0)).sum()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("=" * 60)
print("LLM Metaphor Extraction — Evaluation on 5 Interviews")
print("=" * 60)
print(f"\nGround truth: {y_true.sum()} metaphor segments / {len(y_true)} total")
print(f"LLM predicted: {y_pred.sum()} segments as metaphor")
print(f"\n{'Metric':<20} {'Value':>10}")
print("-" * 32)
print(f"{'TP':<20} {tp:>10}")
print(f"{'FP':<20} {fp:>10}")
print(f"{'FN':<20} {fn:>10}")
print(f"{'TN':<20} {tn:>10}")
print(f"{'Precision':<20} {precision:>10.3f}")
print(f"{'Recall':<20} {recall:>10.3f}")
print(f"{'F1':<20} {f1:>10.3f}")

print(f"\n--- Confusion Matrix ---")
labels = [0, 1]
cm = confusion_matrix(y_true, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=["GT=no", "GT=yes"], columns=["Pred=no", "Pred=yes"])
print(cm_df)

print(f"\n--- sklearn classification report ---")
print(classification_report(y_true, y_pred, target_names=["no metaphor", "metaphor"], zero_division=0))

# Cohen's kappa
kappa = cohen_kappa_score(y_true, y_pred)
print(f"Cohen's kappa (LLM vs GT): {kappa:.4f}")


LLM Metaphor Extraction — Evaluation on 5 Interviews

Ground truth: 38 metaphor segments / 4000 total
LLM predicted: 118 segments as metaphor

Metric                    Value
--------------------------------
TP                            5
FP                          113
FN                           33
TN                         3849
Precision                 0.042
Recall                    0.132
F1                        0.064

--- Confusion Matrix ---
        Pred=no  Pred=yes
GT=no      3849       113
GT=yes       33         5

--- sklearn classification report ---
              precision    recall  f1-score   support

 no metaphor       0.99      0.97      0.98      3962
    metaphor       0.04      0.13      0.06        38

    accuracy                           0.96      4000
   macro avg       0.52      0.55      0.52      4000
weighted avg       0.98      0.96      0.97      4000

Cohen's kappa (LLM vs GT): 0.0505


In [53]:
# === PER-FILE BREAKDOWN ===
print(f"{'File':<50} {'GT+':>4} {'Pred+':>6} {'TP':>4} {'FP':>4} {'FN':>4} {'Prec':>6} {'Rec':>6}")
print("-" * 90)
for f in annotated_files:
    mask = df_5['file'] == f
    gt = df_5.loc[mask, 'y_true_r1'].sum()
    pred = df_5.loc[mask, 'y_pred'].sum()
    t = ((df_5.loc[mask, 'y_true_r1'] == 1) & (df_5.loc[mask, 'y_pred'] == 1)).sum()
    fp_f = ((df_5.loc[mask, 'y_true_r1'] == 0) & (df_5.loc[mask, 'y_pred'] == 1)).sum()
    fn_f = ((df_5.loc[mask, 'y_true_r1'] == 1) & (df_5.loc[mask, 'y_pred'] == 0)).sum()
    p = t / (t + fp_f) if (t + fp_f) > 0 else 0
    r = t / (t + fn_f) if (t + fn_f) > 0 else 0
    short = f[:48]
    print(f"{short:<50} {gt:>4} {pred:>6} {t:>4} {fp_f:>4} {fn_f:>4} {p:>6.2f} {r:>6.2f}")


File                                                GT+  Pred+   TP   FP   FN   Prec    Rec
------------------------------------------------------------------------------------------
ID 37-transcription_anonymiseret.docx                17     43    3   40   14   0.07   0.18
ID 66-transcription_anonymiseret.docx                 6     24    1   23    5   0.04   0.17
ID 09-transcription_edit færdig.docx                  9     22    0   22    9   0.00   0.00
ID 03-transcription_edit færdig.docx                  3     26    1   25    2   0.04   0.33
ID 62-transcription_anonymieret.docx                  3      3    0    3    3   0.00   0.00


In [54]:
# === FALSE NEGATIVES: Ground truth metaphors MISSED by the LLM ===
fn_df = df_5[(df_5['y_true_r1'] == 1) & (df_5['y_pred'] == 0)].copy()
print(f"Total missed metaphors (FN): {len(fn_df)}\n")
print(f"{'#':<3} {'File':<15} {'SegID':>6}  Text")
print("-" * 100)
for i, (idx, row) in enumerate(fn_df.iterrows(), 1):
    fshort = row['file'][:13]
    sid = str(row['segment_id'])[-6:]
    txt = str(row['text']).strip()[:120]
    print(f"{i:<3} {fshort:<15} {sid:>6}  {txt}")


Total missed metaphors (FN): 33

#   File             SegID  Text
----------------------------------------------------------------------------------------------------
1   ID 37-transcr   et_138  Så en strålebehandling laver ikke særlig meget ballade,
2   ID 37-transcr   et_161  Noget, der til gengæld kunne være rigtig godt, og som jeg lige skimmede i dit spørgeskimmer her, det er hvis du ikke kun
3   ID 37-transcr   et_164  Og hvis du kunne bruge det som motivator for måske helt at lægge det på hylden, eller få lagt det over til noget festryg
4   ID 37-transcr   et_242  hvis det er 5 år i helvede
5   ID 37-transcr   et_278  Og hvis man har været på en længere vare, en køretur eller et eller andet, så skal man så en gammel kone.
6   ID 37-transcr   et_346  Så her står du to fluer med et smæk, fordi jeg tænker ikke, der vil være noget nu.
7   ID 37-transcr   et_353  Fordi hvis vi tager noget kalk fra blodet og slipper det ind i knoglerne, så skal vi gerne have, at du er i niveau, og d
8 

In [55]:
# === TRUE POSITIVES: Ground truth metaphors correctly found by LLM ===
tp_df = df_5[(df_5['y_true_r1'] == 1) & (df_5['y_pred'] == 1)].copy()
print(f"True Positives (correctly found): {len(tp_df)}\n")
for i, (idx, row) in enumerate(tp_df.iterrows(), 1):
    print(f"  {i}. [{row['file'][:15]}] {str(row['text']).strip()[:120]}")

# === FALSE POSITIVES: LLM predicted metaphor but GT says no (sample) ===
fp_df = df_5[(df_5['y_true_r1'] == 0) & (df_5['y_pred'] == 1)].copy()
print(f"\n\nFalse Positives (sample — {len(fp_df)} total):\n")
for i, (idx, row) in enumerate(fp_df.head(10).iterrows(), 1):
    print(f"  {i}. [{row['file'][:15]}] {str(row['text']).strip()[:120]}")
if len(fp_df) > 10:
    print(f"  ... and {len(fp_df) - 10} more")


True Positives (correctly found): 5

  1. [ID 37-transcrip] og derfor kan det være en rigtig god investering
  2. [ID 37-transcrip] det er jo også noget, der kommer snigende.
  3. [ID 37-transcrip] det lige der, om man ikke gider have det. Så jeg vil bare sige, at man går fra pære til æble form,
  4. [ID 66-transcrip] det er 51, altså hvad er jeg så lidt pludselig så, wow, nu blomstrer jeg op i min anden ungdom igen, eller fortsætter je
  5. [ID 03-transcrip] Jeg kan ikke lige rigtigt… Men et eller andet bevægelse gør, at det er som om, de stikker med en kniv eller noget skarpt


False Positives (sample — 113 total):

  1. [ID 37-transcrip] så det er jo en ganske lille knude
  2. [ID 37-transcrip] Ja.
  3. [ID 37-transcrip] Ja.
  4. [ID 37-transcrip] Men det er sådan med strålebehandling og tobaksrygning, at strålerne virker rigtig godt, hvis der er meget ild til stede
  5. [ID 37-transcrip] så er det måske ikke så god en investering
  6. [ID 37-transcrip] Fordi det er der, at man kan 

### Summary: LLM Metaphor Extraction on 5 Interview Transcripts

| Metric | Value |
|--------|-------|
| Ground truth metaphor segments | 38 / 4000 |
| LLM predicted segments | 118 |
| **Precision** | **0.042** (5/118) |
| **Recall** | **0.132** (5/38) |
| **F1** | **0.064** |
| Cohen's kappa | 0.05 |

**Key findings:**
- **Very low recall (13.2%)**: The LLM missed 33 out of 38 ground truth metaphors
- **Very low precision (4.2%)**: 113 of 118 LLM predictions were false positives (ground truth says no metaphor)
- The LLM over-predicts: it flags ~3x more segments than the ground truth positives
- Both human raters agreed 100% on the 5 files, so ground truth is robust
- Cohen's kappa ≈ 0.05 indicates near-chance agreement between LLM and human raters

**Note on matching methodology:** The LLM was run on merged turns (consecutive same-speaker segments merged). Matching back to individual ground truth segments was done via text containment and metaphor span substring matching. Some edge cases may exist where a metaphor spans a segment boundary.
